# 01 — DataFrame Basics

Core DataFrame operations: reading a table, projection, filtering, sorting, actions, and schema introspection. All operations are **lazy** until an action triggers SQL execution.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Read a table

`session.table("vendas")` returns a lazy `IrisDataFrame`.

In [ ]:
df = session.table("vendas")
print(type(df).__name__)

## 2. Projection — `select` / `selectExpr`

In [ ]:
df.select("cidade", "estado", "valor").show()

df.selectExpr("cidade", "valor * 1.1 AS valor_com_imposto").show()

## 3. Filtering — `filter` / `where`

In [ ]:
df.filter("estado = 'SP'").show()

df.where(df["valor"] > 150).show()

## 4. Derived columns — `withColumn`

In [ ]:
from irispark.functions import col, round as rnd

df.withColumn("valor_dobro", col("valor") * 2).show()
df.withColumn("valor_arred", rnd(col("valor"), 0)).show()

## 5. Sorting — `orderBy` / `order_by` / `sort`

In [ ]:
df.orderBy("valor DESC").show()
df.order_by(col("valor").desc()).show()

## 6. Limit

In [ ]:
df.limit(3).show()

## 7. Actions

Actions trigger SQL execution and return results.

In [ ]:
print("count:", df.count())
print("first:", df.first())
print("head(2):", df.head(2))
print("take(3):", df.take(3))
print("tail(2):", df.tail(2))
print("isEmpty:", df.isEmpty())

## 8. Interop — `to_pandas` / `toPandas`

In [ ]:
pdf = df.to_pandas()
print(type(pdf).__name__)
pdf.head()

## 9. Schema introspection

In [ ]:
df.printSchema()
print("dtypes:", df.dtypes)
print("schema:", df.schema)

## 10. Summary statistics — `describe` / `summary`

In [ ]:
df.describe().show()
df.summary().show()

## 11. SQL transparency

`df.to_sql()` shows the generated SQL.

In [ ]:
print(df.filter("estado = 'SP'").select("cidade", "valor").to_sql())

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")